# 🦅 EarlyBird + M1 (view-aware gated fusion) – Wildtrack
Code lấy từ fork **nguyenquangpen/NCKH_EarlyBird** (`models/mvdet.py`, tham số `model.view_fusion`).
Mỗi lần **Save & Run All** = 1 lần chạy. Chỉ cần sửa cell cấu hình bên dưới.

| `MODE` | Làm gì | Thời gian |
|---|---|---|
| `smoke` | Unit test + fit 1 epoch, `limit_train_batches=20`, lần lượt cả 3 arm → bảng bộ nhớ GPU | ~15 phút |
| `full` | Unit test + train `EPOCHS` epoch cho 1 `ARM` × 1 `SEED`, test `best` và `last`, rồi sweep ngưỡng trên `last` | ~5 giờ |
| `sweep` | Không train. Test checkpoint `last` của từng arm ở nhiều `conf_threshold` → bảng so sánh | ~45 phút |

| `ARM` | Config | Ý nghĩa |
|---|---|---|
| `none` | `configs/m_none.yml` | Baseline EarlyBird, kiến trúc y hệt code gốc |
| `geo` | `configs/m_geo.yml` | w = prior hình học (blur visibility) chuẩn hoá trên camera, 0 tham số học, alpha = 1 |
| `m1` | `configs/m_m1.yml` | w = softmax(log W_geo + g([F_s, ray_s])), gate = 1 + alpha·(N·w − 1), alpha khởi tạo 0 |

Effective batch luôn = 8 như cấu hình gốc (`t_fit.yml`: 1 GPU × accumulate 8): với 2 GPU notebook tự đặt `accumulate_grad_batches = 4`.

**Settings:** GPU T4 x2 · Internet On · Input: dataset Wildtrack

## 1. Cấu hình & cài đặt

In [ ]:
FORK_URL = "https://github.com/nguyenquangpen/NCKH_EarlyBird.git"
BRANCH   = "main"
MODE     = "smoke"  # smoke | full | sweep
ARM      = "m1"     # none | geo | m1          (chỉ dùng khi MODE = "full")
SEED     = 1337     # 1337 | 42 | 2024
EPOCHS   = 50
NUM_GPUS = 2        # full: DDP 2×T4. smoke / sweep luôn dùng 1 GPU
ACCUM    = 8 // NUM_GPUS   # giữ effective batch = 8 như cấu hình gốc (1 GPU × accumulate 8)
# sweep: checkpoint "<arm>_s<SEED>_last.ckpt" của từng arm phải có trong /kaggle/input (Add Input)
SWEEP_ARMS = ["none", "geo", "m1"]
SWEEP_THRS = [0.40, 0.45, 0.50, 0.55, 0.60]
RUN      = f"{ARM}_s{SEED}" if MODE == "full" else f"{MODE}_s{SEED}"

assert MODE in ("smoke", "full", "sweep") and ARM in ("none", "geo", "m1")
%cd /kaggle/working
!rm -rf /kaggle/working/EarlyBird
!git clone -q -b {BRANCH} {FORK_URL} /kaggle/working/EarlyBird
!git -C /kaggle/working/EarlyBird log -1 --format="Commit: %h  %s"
!pip install -q "setuptools<70" "numpy<2" "lightning==2.5.5" "jsonargparse[signatures]>=4.39.0" \
    "tensorboard==2.16.2" "protobuf==3.20.3" "kornia==0.6.11" "timm==0.9.2" "einops==0.6.1" \
    "pytorch-metric-learning==2.2.0" efficientnet_pytorch "lap>=0.5" "motmetrics==1.4.0" "nuscenes-devkit==1.2.0" pytest
%cd /kaggle/working/EarlyBird/EarlyBird
import torch
print("torch", torch.__version__, "| CUDA build:", torch.version.cuda, "| GPU:", torch.cuda.device_count())
need = NUM_GPUS if MODE == "full" else 1
assert torch.version.cuda and torch.cuda.device_count() >= need, (
    f"Cần {need} GPU nhưng đang có {torch.cuda.device_count()} (torch CUDA build: {torch.version.cuda}). "
    "Bật Settings → Accelerator → GPU T4 x2 rồi chạy lại.")
print("RUN =", RUN)

## 2. Unit test (CPU)
`tests/test_view_fusion.py`: tổng w trên camera nhìn thấy = 1, camera không thấy = 0; m1 với alpha = 0 giữ nguyên F_s;
shape đúng cho cả 3 arm; `none` cho output giống hệt code gốc fee7379. Lỗi ở đây thì dừng, không train.

In [ ]:
import subprocess
r = subprocess.run("python -m pytest tests/test_view_fusion.py -q -p no:warnings", shell=True,
                   capture_output=True, text=True, env={**__import__("os").environ, "CUDA_VISIBLE_DEVICES": ""})
print(r.stdout[-3000:], r.stderr[-2000:])
assert r.returncode == 0, "Unit test lỗi – xem log ở trên"
for arm in ("none", "geo", "m1"):
    print(f"configs/m_{arm}.yml:", open(f"configs/m_{arm}.yml").read().strip().replace("\n", " "))

## 3. Dữ liệu

In [ ]:
import glob, os, shutil, json
src = os.path.dirname(os.path.dirname(os.path.dirname(
    glob.glob("/kaggle/input/**/calibrations/extrinsic/extr_CVLab1.xml", recursive=True)[0])))
dst = "/kaggle/working/Wildtrack"  # tên thư mục phải là "Wildtrack" (datamodule nhận dạng theo basename)
shutil.rmtree(dst, ignore_errors=True); os.makedirs(dst)
for name in ["Image_subsets", "annotations_positions", "calibrations"]:
    os.symlink(f"{src}/{name}", f"{dst}/{name}")
print(src, "→", dst)

## 4. Config, callback, hàm chạy

In [ ]:
%%writefile configs/t_kaggle.yml
trainer:
  accelerator: gpu
  devices: 2
  strategy:
    class_path: lightning.pytorch.strategies.DDPStrategy
    init_args: {find_unused_parameters: true}
  max_epochs: 50
  accumulate_grad_batches: 8
  gradient_clip_val: 0.5
  check_val_every_n_epoch: 1
  log_every_n_steps: 5
  enable_progress_bar: false
  logger:
    - class_path: lightning.pytorch.loggers.TensorBoardLogger
      init_args: {save_dir: /kaggle/working/runs, name: tb}
    - class_path: lightning.pytorch.loggers.CSVLogger
      init_args: {save_dir: /kaggle/working/runs, name: csv}
  callbacks:
    - class_path: lightning.pytorch.callbacks.LearningRateMonitor
      init_args: {logging_interval: step}
    - class_path: lightning.pytorch.callbacks.ModelCheckpoint
      init_args:
        dirpath: /kaggle/working/checkpoints
        monitor: val_loss
        mode: min
        save_last: true
        filename: "model-{epoch:02d}-{val_loss:.2f}"
    - class_path: kaggle_utils.Progress
data:
  init_args:
    data_dir: /kaggle/working/Wildtrack
    num_workers: 4

In [ ]:
%%writefile configs/t_kaggle_test.yml
trainer:
  accelerator: gpu
  devices: 1
  enable_progress_bar: false
  logger:
    class_path: lightning.pytorch.loggers.CSVLogger
    init_args: {save_dir: /kaggle/working/runs, name: test}
  callbacks:
    - class_path: kaggle_utils.Progress
data:
  init_args:
    data_dir: /kaggle/working/Wildtrack
    num_workers: 4

In [ ]:
%%writefile kaggle_utils.py
"""Gửi tiến độ train/test của EarlyBird ra stdout dạng sự kiện '@@EB {json}'.
Notebook đọc các sự kiện này để vẽ thanh tqdm và bảng kết quả. Không can thiệp vào việc train.
Với view_fusion = m1 còn ghi lại alpha (gate) sau mỗi epoch."""
import json, math, os, time
import torch
from lightning.pytorch.callbacks import Callback

OUT = "/kaggle/working/results"


def _f(x):
    try:
        v = float(x.item() if hasattr(x, "item") else x)
        return v if math.isfinite(v) else None
    except Exception:
        return None


def emit(kind, **d):
    print("@@EB " + json.dumps({"e": kind, **d}), flush=True)


def _fusion_info(pl_module):
    fusion = getattr(pl_module.model, "fusion", None)
    if fusion is None:
        return {"view_fusion": getattr(pl_module.model, "view_fusion", "none"), "alpha": None, "fusion_params": 0}
    return {"view_fusion": fusion.mode, "alpha": _f(fusion.alpha),
            "fusion_params": sum(p.numel() for p in fusion.parameters())}


class Progress(Callback):
    def __init__(self):
        super().__init__()
        self.t0 = self.ep_t0 = time.time()
        self.losses, self.history = [], []

    def on_sanity_check_start(self, trainer, pl_module):
        if trainer.is_global_zero:
            emit("status", msg="Sanity check validation...")

    def on_train_start(self, trainer, pl_module):
        if not trainer.is_global_zero:
            return
        self.t0 = time.time()
        bs = getattr(trainer.datamodule, "batch_size", 1)
        emit("fit_start", max_epochs=trainer.max_epochs, start_epoch=trainer.current_epoch,
             steps=trainer.num_training_batches, world=trainer.world_size,
             eff_batch=bs * trainer.accumulate_grad_batches * trainer.world_size,
             gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
             params=sum(p.numel() for p in pl_module.parameters()), **_fusion_info(pl_module))

    def on_train_epoch_start(self, trainer, pl_module):
        if trainer.is_global_zero:
            self.ep_t0, self.losses = time.time(), []
            emit("phase", name="train", epoch=trainer.current_epoch, total=trainer.num_training_batches)
            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        if trainer.is_global_zero:
            v = _f(trainer.callback_metrics.get("train_loss"))
            if v is not None:
                self.losses.append(v)
            emit("step", loss=sum(self.losses) / max(len(self.losses), 1))

    def on_validation_epoch_start(self, trainer, pl_module):
        if trainer.is_global_zero and not trainer.sanity_checking:
            emit("phase", name="val", epoch=trainer.current_epoch, total=int(sum(trainer.num_val_batches)))

    def on_validation_batch_end(self, trainer, pl_module, outputs, batch, batch_idx, dataloader_idx=0):
        if trainer.is_global_zero and not trainer.sanity_checking:
            emit("step")

    def on_train_epoch_end(self, trainer, pl_module):  # Lightning 2.x: chạy SAU validation
        if not trainer.is_global_zero:
            return
        cm = trainer.callback_metrics
        row = {"epoch": trainer.current_epoch + 1,
               "train_loss": sum(self.losses) / len(self.losses) if self.losses else None,
               "val_loss": _f(cm.get("val_loss")),
               "lr": trainer.optimizers[0].param_groups[0]["lr"],
               "time": time.time() - self.ep_t0,
               "mem": torch.cuda.max_memory_allocated() / 2 ** 30 if torch.cuda.is_available() else 0,
               "mem_reserved": torch.cuda.max_memory_reserved() / 2 ** 30 if torch.cuda.is_available() else 0,
               "alpha": _fusion_info(pl_module)["alpha"]}
        self.history.append(row)
        emit("epoch_end", **row)

    def on_train_end(self, trainer, pl_module):
        if not trainer.is_global_zero:
            return
        ck = trainer.checkpoint_callback
        s = {"best_model_path": ck.best_model_path, "best_val_loss": _f(ck.best_model_score),
             "last_model_path": ck.last_model_path, "hours": (time.time() - self.t0) / 3600,
             **_fusion_info(pl_module), "history": self.history}
        os.makedirs(OUT, exist_ok=True)
        json.dump(s, open(f"{OUT}/train_summary.json", "w"), indent=2)
        emit("fit_end", **{k: v for k, v in s.items() if k != "history"})

    def on_test_epoch_start(self, trainer, pl_module):
        if trainer.is_global_zero:
            emit("phase", name="test", epoch=0, total=int(sum(trainer.num_test_batches)))

    def on_test_batch_end(self, trainer, pl_module, outputs, batch, batch_idx, dataloader_idx=0):
        if trainer.is_global_zero:
            emit("step")

    def on_test_end(self, trainer, pl_module):
        if trainer.is_global_zero:
            m = {k: _f(v) for k, v in trainer.callback_metrics.items()}
            m.update({k: v for k, v in _fusion_info(pl_module).items()})
            os.makedirs(OUT, exist_ok=True)
            json.dump(m, open(f"{OUT}/test_metrics.json", "w"), indent=2)
            emit("test_end")

In [ ]:
import json, os, subprocess, time
from tqdm.auto import tqdm

ENV = {**os.environ, "PYTHONUNBUFFERED": "1", "TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD": "1",
       "PYTHONWARNINGS": "ignore::UserWarning,ignore::FutureWarning,ignore::DeprecationWarning"}
HEAD = (f"{'Epoch':>7} │ {'train_loss':>10} │ {'val_loss':>10} │ {'lr':>9} │ {'thời gian':>9} │ "
        f"{'GPU':>6} │ {'alpha':>7}")
RES = "/kaggle/working/results"


def run(args, log_path):
    """Chạy `python main.py <args>`, vẽ tqdm + bảng epoch, ghi log đầy đủ vào log_path."""
    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    print(f"⏳ python main.py {args}\n   Đang load dữ liệu & khởi tạo model (1–2 phút)...")
    p = subprocess.Popen(f"python main.py {args}", shell=True, cwd="/kaggle/working/EarlyBird/EarlyBird",
                         env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    ep_bar = bar = None
    best = float("inf")
    with open(log_path, "w") as log:
        for line in p.stdout:
            log.write(line)
            if not line.startswith("@@EB "):
                if line.strip() and bar is None:      # trước khi train: in log khởi tạo bình thường
                    print(line.rstrip())
                elif "Error" in line or "Traceback" in line:
                    tqdm.write(line.rstrip())
                continue
            ev = json.loads(line[5:])
            e = ev["e"]
            if e == "status":
                print("🔎", ev["msg"])
            elif e == "fit_start":
                print(f"\n🚀 Train trên {ev['world']}× {ev['gpu']} | effective batch {ev['eff_batch']} | "
                      f"{ev['steps']} batch/epoch/GPU")
                print(f"   view_fusion = {ev['view_fusion']} | tham số fusion = {ev['fusion_params']:,} | "
                      f"tổng tham số = {ev['params']:,}\n")
                ep_bar = tqdm(total=ev["max_epochs"], initial=ev["start_epoch"], desc="Tổng", unit="epoch",
                              position=0, dynamic_ncols=True)
                bar = tqdm(total=ev["steps"], desc="Epoch", position=1, dynamic_ncols=True)
                tqdm.write(HEAD + "\n" + "─" * len(HEAD))
            elif e == "phase":
                if bar is None:
                    bar = tqdm(total=ev["total"], position=0, dynamic_ncols=True)
                total = ep_bar.total if ep_bar else 0
                label = {"train": f"Epoch {ev['epoch'] + 1}/{total} · train",
                         "val": f"Epoch {ev['epoch'] + 1}/{total} · val",
                         "test": "Test"}[ev["name"]]
                bar.reset(total=ev["total"])
                bar.set_description(label)
                bar.set_postfix_str("")
            elif e == "step":
                bar.update(1)
                if ev.get("loss") is not None:
                    bar.set_postfix(loss=f"{ev['loss']:.4f}", refresh=False)
            elif e == "epoch_end":
                v = ev["val_loss"]
                star = ""
                if v is not None and v < best:
                    best, star = v, "  ★"
                f = lambda x: f"{x:>10.4f}" if x is not None else f"{'-':>10}"
                a = f"{ev['alpha']:>7.4f}" if ev.get("alpha") is not None else f"{'-':>7}"
                tqdm.write(f"{ev['epoch']:>7} │ {f(ev['train_loss'])} │ {f(v)} │ {ev['lr']:>9.2e} │ "
                           f"{time.strftime('%M:%S', time.gmtime(ev['time'])):>9} │ {ev['mem']:>5.2f}G │ {a}{star}")
                ep_bar.update(1)
                ep_bar.set_postfix(best_val=f"{best:.4f}")
            elif e == "fit_end":
                for b in (bar, ep_bar):
                    b and b.close()
                bv = ev["best_val_loss"]
                print(f"\n✅ Xong sau {ev['hours']:.2f} giờ | best val_loss = "
                      f"{bv:.4f}" if bv is not None else f"\n✅ Xong sau {ev['hours']:.2f} giờ")
                print(f"   {ev['best_model_path']}")
            elif e == "test_end":
                bar and bar.close()
                print("✅ Test xong")
    p.wait()
    for b in (bar, ep_bar):
        b and b.close()
    if p.returncode != 0:
        tail = [l for l in open(log_path).read().splitlines() if not l.startswith("@@EB ")][-40:]
        print("\n".join(tail))
        raise RuntimeError(f"Lỗi (exit {p.returncode}) – log đầy đủ: {log_path}")


def clean():
    for d in ["/kaggle/working/runs", "/kaggle/working/checkpoints"]:
        shutil.rmtree(d, ignore_errors=True)


def common(arm):
    return f"-c configs/d_wildtrack.yml -c configs/m_{arm}.yml --seed_everything {SEED}"


SWEEP_METRICS = [("MODA", "detect/moda", 1), ("Precision", "detect/precision", 1), ("Recall", "detect/recall", 1),
                 ("IDF1", "track/idf1", 100), ("MOTA", "track/mota", 100), ("FP", "track/num_false_positives", 1),
                 ("Miss", "track/num_misses", 1), ("ID sw", "track/num_switches", 1)]


def sweep_ckpt(arm, ckpt, tag):
    """Test `ckpt` at every conf_threshold in SWEEP_THRS -> {"0.40": metrics, ...}.
    LightningCLI re-applies the hyper_parameters stored in --ckpt_path on top of the command line, so
    --model.conf_threshold alone is silently reset to the trained value (0.4): the threshold is written into
    a temporary copy of the checkpoint instead."""
    import torch
    tmp = "/kaggle/working/sweep_tmp.ckpt"
    state = torch.load(ckpt, map_location="cpu", weights_only=False)
    out = {}
    for thr in SWEEP_THRS:
        print(f"\n===== sweep {tag} | conf_threshold = {thr:.2f} =====")
        state["hyper_parameters"]["conf_threshold"] = thr
        torch.save(state, tmp)
        run(f'test {common(arm)} -c configs/t_kaggle_test.yml --model.conf_threshold {thr} --ckpt_path "{tmp}"',
            f"{RES}/sweep_{tag}_{thr:.2f}.log")
        os.replace(f"{RES}/test_metrics.json", f"{RES}/sweep_{tag}_{thr:.2f}.json")
        out[f"{thr:.2f}"] = json.load(open(f"{RES}/sweep_{tag}_{thr:.2f}.json"))
    del state
    os.remove(tmp)
    seen = {(m.get("track/num_false_positives"), m.get("track/num_misses")) for m in out.values()}
    assert len(SWEEP_THRS) == 1 or len(seen) > 1, f"{tag}: mọi ngưỡng cho kết quả giống hệt nhau – ngưỡng không được áp dụng"
    return out


def print_sweep(sweep):
    """sweep: {arm: {"0.40": metrics, ...}} -> one table per metric, best threshold and mean over thresholds."""
    thrs = list(next(iter(sweep.values())))
    for name, k, sc in SWEEP_METRICS:
        better = min if name in ("FP", "Miss", "ID sw") else max
        print(f"\n{name:<9}│" + "".join(f"{t:>8}" for t in thrs) + f" │ {'mean':>6} │ {'tốt nhất':>13}")
        for arm, res in sweep.items():
            vals = [(res[t].get(k) or 0) * sc for t in thrs]
            b = better(range(len(vals)), key=lambda i: vals[i])
            print(f"{arm:<9}│" + "".join(f"{v:>8.2f}" for v in vals)
                  + f" │ {sum(vals) / len(vals):>6.2f} │ {vals[b]:>7.2f} @{thrs[b]}")

## 5a. Smoke test (`MODE = "smoke"`)
Fit 1 epoch, `limit_train_batches=20`, `limit_val_batches=5`, 1 GPU, lần lượt `none → geo → m1`.
Kiểm tra không lỗi và đo bộ nhớ GPU đỉnh (allocated / reserved) của epoch train.

In [ ]:
if MODE == "smoke":
    shutil.rmtree(RES, ignore_errors=True)
    smoke = {}
    for arm in ("none", "geo", "m1"):
        print(f"\n{'=' * 20} smoke: {arm} {'=' * 20}")
        clean()
        run(f"fit {common(arm)} -c configs/t_kaggle.yml --trainer.max_epochs 1 --trainer.devices 1 "
            f"--trainer.strategy auto --trainer.limit_train_batches 20 --trainer.limit_val_batches 5",
            f"{RES}/smoke_{arm}.log")
        s = json.load(open(f"{RES}/train_summary.json"))
        os.replace(f"{RES}/train_summary.json", f"{RES}/smoke_summary_{arm}.json")
        h = s["history"][-1]
        smoke[arm] = {"fusion_params": s["fusion_params"], "mem_alloc_G": h["mem"],
                      "mem_reserved_G": h["mem_reserved"], "sec_per_epoch": h["time"],
                      "train_loss": h["train_loss"], "val_loss": h["val_loss"], "alpha": h["alpha"]}
    clean()
    base = smoke["none"]["mem_alloc_G"]
    print(f"\n{'arm':<5} │ {'fusion params':>13} │ {'GPU alloc':>9} │ {'Δ vs none':>9} │ {'reserved':>8} │ "
          f"{'s/epoch':>7} │ {'train_loss':>10} │ {'val_loss':>8}\n" + "─" * 92)
    for arm, r in smoke.items():
        vl = f"{r['val_loss']:>8.3f}" if r["val_loss"] is not None else f"{'-':>8}"
        print(f"{arm:<5} │ {r['fusion_params']:>13,} │ {r['mem_alloc_G']:>8.2f}G │ {r['mem_alloc_G'] - base:>+8.2f}G │ "
              f"{r['mem_reserved_G']:>7.2f}G │ {r['sec_per_epoch']:>7.0f} │ {r['train_loss'] or float('nan'):>10.3f} │ {vl}")
    json.dump(smoke, open(f"{RES}/smoke_{SEED}.json", "w"), indent=2)
    shutil.make_archive(f"/kaggle/working/results_{RUN}", "zip", RES)
    print("\n📦", f"results_{RUN}.zip  –  smoke xong, các cell bên dưới chỉ chạy khi MODE = 'full'")

## 5b. Train (`MODE = "full"`)

In [ ]:
if MODE == "full":
    shutil.rmtree(RES, ignore_errors=True)
    clean()
    run(f"fit {common(ARM)} -c configs/t_kaggle.yml --trainer.max_epochs {EPOCHS} --trainer.devices {NUM_GPUS} "
        f"--trainer.accumulate_grad_batches {ACCUM}",
        f"{RES}/train.log")

## 6. Test cả `best` (cách tác giả) và `last` (công bằng)

In [ ]:
if MODE == "full":
    summ = json.load(open(f"{RES}/train_summary.json"))
    for tag, ckpt in [("best", summ["best_model_path"]), ("last", "/kaggle/working/checkpoints/last.ckpt")]:
        print(f"\n===== Test {tag}: {os.path.basename(ckpt)} =====")
        run(f'test {common(ARM)} -c configs/t_kaggle_test.yml --ckpt_path "{ckpt}"', f"{RES}/test_{tag}.log")
        os.replace(f"{RES}/test_metrics.json", f"{RES}/test_metrics_{tag}.json")
        pred_dir = sorted(glob.glob("/kaggle/working/runs/test/version_*"), key=os.path.getmtime)[-1]
        os.makedirs(f"{RES}/pred_{tag}", exist_ok=True)
        for f in glob.glob(f"{pred_dir}/*.txt"):
            shutil.copy(f, f"{RES}/pred_{tag}/")

## 6b. Sweep ngưỡng `conf_threshold` (`MODE = "full"` và `"sweep"`)
Test checkpoint `last` ở từng ngưỡng trong `SWEEP_THRS` (ngưỡng dùng cho cả MODA lẫn tracker), để so các arm
**ở cùng mức precision / ngưỡng tốt nhất của chính nó** chứ không chỉ ở 0.4 – M1 đẩy recall lên nên ngưỡng 0.4 thiệt cho nó.
Lưu ý: tập test = tập val của Wildtrack, nên "ngưỡng tốt nhất" lạc quan như nhau cho mọi arm; con số công bằng hơn là cột `mean`.

- `full`: tự chạy sau khi train, trên `/kaggle/working/checkpoints/last.ckpt` (~15 phút).
- `sweep`: không train. Cần Kaggle Dataset chứa `<arm>_s<SEED>_last.ckpt` cho các arm trong `SWEEP_ARMS`
  (đuôi phải là **.ckpt** – nếu để .zip Kaggle sẽ tự giải nén và hỏng file), rồi **Add Input** dataset đó.

In [ ]:
sweep = {}
if MODE == "full":
    sweep[ARM] = sweep_ckpt(ARM, "/kaggle/working/checkpoints/last.ckpt", f"{ARM}_last")
elif MODE == "sweep":
    shutil.rmtree(RES, ignore_errors=True)
    clean()
    for arm in SWEEP_ARMS:
        hits = [h for h in glob.glob(f"/kaggle/input/**/{arm}_s{SEED}_last*", recursive=True) if os.path.isfile(h)]
        if not hits:
            print(f"⚠️ Không thấy file {arm}_s{SEED}_last.ckpt trong /kaggle/input – bỏ qua arm này")
            continue
        print(f"{arm:<5} → {sorted(hits)[0]}")
        sweep[arm] = sweep_ckpt(arm, sorted(hits)[0], arm)
    assert sweep, "Chưa Add Input dataset chứa checkpoint"

if sweep:
    SWEEP_JSON = f"{RES}/sweep_s{SEED}.json" if MODE == "sweep" else f"{RES}/sweep_{RUN}.json"
    json.dump({f"{a}@{t}": m for a, res in sweep.items() for t, m in res.items()}, open(SWEEP_JSON, "w"), indent=2)
    print_sweep(sweep)
if MODE == "sweep":
    shutil.make_archive(f"/kaggle/working/results_{RUN}", "zip", RES)
    print("\n📦", f"results_{RUN}.zip")

## 7. Kết quả
Chạy trong process riêng để tránh lỗi numpy của kernel. Với `m1` có thêm đồ thị alpha theo epoch.

In [ ]:
%%writefile /kaggle/working/report.py
import csv, glob, json, os, subprocess, sys
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
RES, RUN = "/kaggle/working/results", sys.argv[1]
summ = json.load(open(f"{RES}/train_summary.json"))

rows = list(csv.DictReader(open(sorted(glob.glob("/kaggle/working/runs/csv/version_*/metrics.csv"))[-1])))
num = lambda r, k: float(r[k]) if r.get(k) not in (None, "") else None
tr = [(num(r, "step"), num(r, "train_loss")) for r in rows if num(r, "train_loss") is not None]
va = {int(num(r, "epoch")) + 1: num(r, "val_loss") for r in rows if num(r, "val_loss") is not None}
alphas = [(h["epoch"], h["alpha"]) for h in summ["history"] if h.get("alpha") is not None]
fig, ax = plt.subplots(1, 3 if alphas else 2, figsize=(20 if alphas else 14, 4))
ax[0].plot([x for x, _ in tr], [y for _, y in tr], alpha=.4); ax[0].set(title="Train loss", xlabel="step")
ax[1].plot(list(va), list(va.values()), marker="o", ms=3); ax[1].set(title="Val loss", xlabel="epoch")
if alphas:
    ax[2].plot(*zip(*alphas), marker="o", ms=3); ax[2].set(title="M1 gate alpha", xlabel="epoch")
fig.suptitle(RUN); plt.tight_layout(); fig.savefig(f"{RES}/curves.png", dpi=120)

M = {t: json.load(open(f"{RES}/test_metrics_{t}.json")) for t in ["best", "last"]}
keys = [("MODA", "detect/moda", 1), ("MODP", "detect/modp", 1), ("Precision", "detect/precision", 1),
        ("Recall", "detect/recall", 1), ("IDF1", "track/idf1", 100), ("MOTA", "track/mota", 100),
        ("ID switches", "track/num_switches", 1)]
print(f"\n{RUN}  (view_fusion = {summ['view_fusion']}, fusion params = {summ['fusion_params']:,})")
print(f"{'Metric':<12} │ {'best':>8} │ {'last':>8}\n" + "─" * 34)
table = {}
for name, k, s in keys:
    b, l = (M[t].get(k) for t in ["best", "last"])
    table[name] = {"best": b * s if b is not None else None, "last": l * s if l is not None else None}
    f = lambda v: f"{v:>8.2f}" if v is not None else f"{'–':>8}"
    print(f"{name:<12} │ {f(table[name]['best'])} │ {f(table[name]['last'])}")
alpha = {t: M[t].get("alpha") for t in ["best", "last"]}
if summ["view_fusion"] == "m1":
    print(f"{'alpha':<12} │ {alpha['best']:>8.4f} │ {alpha['last']:>8.4f}")
sweep, sweep_mean = {}, {}
if os.path.exists(f"{RES}/sweep_{RUN}.json"):
    raw = json.load(open(f"{RES}/sweep_{RUN}.json"))
    S = [("MODA", "detect/moda", 1), ("Precision", "detect/precision", 1), ("Recall", "detect/recall", 1),
         ("IDF1", "track/idf1", 100), ("MOTA", "track/mota", 100), ("FP", "track/num_false_positives", 1),
         ("Miss", "track/num_misses", 1)]
    thrs = [k.split("@")[1] for k in raw]
    sweep = {t: {n: (raw[f"{RUN.rsplit('_s', 1)[0]}@{t}"].get(k) or 0) * s for n, k, s in S} for t in thrs}
    sweep_mean = {n: sum(sweep[t][n] for t in thrs) / len(thrs) for n, _, _ in S}
    print(f"\nSweep conf_threshold (last)\n{'thr':<6} │" + "".join(f"{n:>10}" for n, _, _ in S) + "\n" + "─" * 80)
    for t in thrs:
        print(f"{t:<6} │" + "".join(f"{sweep[t][n]:>10.2f}" for n, _, _ in S))
    print(f"{'mean':<6} │" + "".join(f"{sweep_mean[n]:>10.2f}" for n, _, _ in S))
peak = max(h["mem"] for h in summ["history"])
print(f"\nGPU peak (allocated, GPU 0): {peak:.2f} GB | train {summ['hours']:.2f} giờ")
commit = subprocess.run(["git", "-C", "/kaggle/working/EarlyBird", "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
json.dump({"run": RUN, "commit": commit, "view_fusion": summ["view_fusion"], "fusion_params": summ["fusion_params"],
           "alpha": alpha, "best_ckpt": summ["best_model_path"], "train_hours": summ["hours"],
           "gpu_peak_G": peak, "metrics": table, "sweep_last": sweep, "sweep_last_mean": sweep_mean}, open(f"{RES}/summary_{RUN}.json", "w"), indent=2)

In [ ]:
if MODE == "full":
    from IPython.display import Image, display
    !python /kaggle/working/report.py {RUN}
    display(Image(f"{RES}/curves.png"))
    shutil.copy("/kaggle/working/checkpoints/last.ckpt", f"/kaggle/working/{RUN}_last.ckpt")
    shutil.copy(summ["best_model_path"], f"/kaggle/working/{RUN}_best.ckpt")
    shutil.make_archive(f"/kaggle/working/results_{RUN}", "zip", RES)
    print("📦", f"results_{RUN}.zip")